In [ ]:
import sys
from pathlib import Path

CURRENT = Path.cwd().resolve()
PROJECT_ROOT = CURRENT
if not (PROJECT_ROOT / "SEConformer").exists():
    for parent in [CURRENT, *CURRENT.parents]:
        if (parent / "SEConformer").exists():
            PROJECT_ROOT = parent
            break

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import torch
print("Projeto:", PROJECT_ROOT)
print("Torch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.is_available())


In [ ]:
%load_ext autoreload
%autoreload 2

import SEConformer as seconformer

BREAKHIS_BASE_PATH = PROJECT_ROOT / "BrestCancer Datasets" / "BreaKHis" / "BreaKHis_v1" / "BreaKHis_v1" / "histology_slides" / "breast"
BREAKHIS_CSV = PROJECT_ROOT / "SEConformer" / "breakhis_binary_folds.csv"
INBREAST_CSV_SOURCE = PROJECT_ROOT / "BrestCancer Datasets" / "INBreast" / "INbreast" / "INbreast.csv"
INBREAST_DICOM_DIR = PROJECT_ROOT / "BrestCancer Datasets" / "INBreast" / "INbreast" / "AllDICOMs"
INBREAST_FOLDS_CSV = PROJECT_ROOT / "SEConformer" / "inbreast_birads_folds.csv"

print("Device:", seconformer.DEVICE)


In [ ]:
seconformer.build_breakhis_csv(
    base_path=str(BREAKHIS_BASE_PATH),
    out_csv=str(BREAKHIS_CSV),
    mode="binary",
    n_splits=5,
    random_state=42,
)

seconformer.build_inbreast_csv(
    inbreast_csv_path=str(INBREAST_CSV_SOURCE),
    dicom_dir=str(INBREAST_DICOM_DIR),
    out_csv=str(INBREAST_FOLDS_CSV),
    mode="multiclass",
    n_splits=5,
    random_state=42,
)


In [ ]:
source_run_dirs = seconformer.make_run_dirs(run_prefix="seconformer_breakhis_source")
source_model, source_run_dirs, source_metrics = seconformer.train_breakhis_holdout(
    base_path=str(BREAKHIS_BASE_PATH),
    mode="binary",
    val_fraction=0.2,
    epochs=5,
    batch_size=16,
    run_dirs=source_run_dirs,
    img_size=224,
    balance=True,
)

print("Run fonte:", source_run_dirs["out_dir"])
source_metrics


In [ ]:
target_run_dirs = seconformer.make_run_dirs(run_prefix="seconformer_breakhis_to_inbreast_tl")
target_model, target_run_dirs, target_metrics = seconformer.train_transfer_breakhis_to_inbreast(
    inbreast_csv_path=str(INBREAST_FOLDS_CSV),
    val_fraction=0.2,
    epochs=5,
    batch_size=16,
    run_dirs=target_run_dirs,
    img_size=224,
    balance=True,
    freeze_backbone=True,
)

print("Run alvo:", target_run_dirs["out_dir"])
target_metrics
